# Kapitel 5 — Klassifizieren und entscheiden · Kommentierte Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch05_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python. Bei jeder Berechnung nennen die Kommentare die Gruppe, die als Nenner dient.

**Inhalt.** Daten · Die Berechnungen der Lektionen 5.1 bis 5.6 (Likelihood, Softmax) · Aufgaben 5.1 bis 5.6

**Konventionen des Notebooks.**
- Entscheidungsregel (Gleichung 5.1): Eine Nachricht wird markiert, wenn ihre geschätzte Wahrscheinlichkeit **größer oder gleich** dem Schwellenwert ist. Für die zwölf Nachrichten wird die Regel auf den **nicht gerundeten** Wert `p_exact` angewendet, der mit der Sigmoidfunktion neu berechnet wird; die Tabelle im Buch zeigt nur den auf zwei Dezimalstellen gerundeten Wert. In den Aufgaben, in denen die Wahrscheinlichkeiten als Dezimalzahlen gegeben sind, sind diese Dezimalzahlen die zu verwendenden Werte.
- Ein Maß, dessen Nenner null ist (zum Beispiel die Präzision, wenn kein Alarm ausgelöst wird), ist `None` und wird als „undefiniert“ angezeigt.
- Der F1 wird als $2\,VP / (2\,VP + FP + FN)$ berechnet (Gleichung 5.8).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe = eine Rolle
SPAM, LEGIT, FLAGGED, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
THRESHOLD, ORANGE, BLUE, GREEN = '#8e24aa', '#eb6834', '#2a78d6', '#2e8b57'   # Schwellenwert; Sigmoidfunktion; Verlust; Filter B


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def show_value(v):
    """Konvention des Notebooks: Ein Maß mit Nenner null ist None und wird als „undefiniert“ angezeigt."""
    return 'undefiniert' if v is None else f'{v:.2f}'

## Daten

Die Kennungen D5.1, D5.2, D5.E1, D5.E2 und D5.E3 sind die des Buchs und der CSV-Dateien.

In [ ]:
# Daten von Kapitel 5, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D5.1 — zwölf Nachrichten: tatsächliches Label, Links, verdächtige Wörter, geschätzte Wahrscheinlichkeit auf zwei Dezimalstellen GERUNDET
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4, 0.95),
    ('M2', 'legitime', 2, 0, 0.12),
    ('M3', 'legitime', 4, 0, 0.50),
    ('M4', 'spam', 3, 3, 0.62),
    ('M5', 'legitime', 1, 0, 0.05),
    ('M6', 'legitime', 2, 2, 0.27),
    ('M7', 'spam', 2, 3, 0.38),
    ('M8', 'legitime', 2, 1, 0.18),
    ('M9', 'spam', 4, 4, 0.88),
    ('M10', 'legitime', 1, 1, 0.08),
    ('M11', 'legitime', 0, 0, 0.02),
    ('M12', 'legitime', 0, 1, 0.03)
], columns=['message', 'label', 'links', 'suspicious_words', 'score'])

# D5.2 — Filter A: 1000 Nachrichten nach Intervall (Anzahl, Spams, mittlere vorhergesagte Wahrscheinlichkeit)
calibration = pd.DataFrame([
    (0.0, 0.1, 540, 22, 0.03),
    (0.1, 0.3, 150, 29, 0.19),
    (0.3, 0.5, 80, 33, 0.40),
    (0.5, 0.7, 60, 35, 0.60),
    (0.7, 0.9, 90, 71, 0.80),
    (0.9, 1.0, 80, 77, 0.95)
], columns=['bin_min', 'bin_max', 'messages', 'spams', 'mean_proba'])

# D5.E1 — Filter B: gleiche Tabelle (Aufgabe 5.1)
calibration_b = pd.DataFrame([
    (0.0, 0.1, 400, 40, 0.02),
    (0.1, 0.3, 175, 42, 0.18),
    (0.3, 0.5, 100, 43, 0.41),
    (0.5, 0.7, 100, 55, 0.61),
    (0.7, 0.9, 150, 93, 0.82),
    (0.9, 1.0, 75, 56, 0.97)
], columns=['bin_min', 'bin_max', 'messages', 'spams', 'mean_proba'])

# D5.E2 — zehn Nachrichten des Vereins (Aufgabe 5.2): Diese Dezimalzahlen sind die zu verwendenden Werte
ten = pd.DataFrame([
    ('N1', 'spam', 0.94),
    ('N2', 'legitime', 0.07),
    ('N3', 'legitime', 0.66),
    ('N4', 'spam', 0.50),
    ('N5', 'legitime', 0.30),
    ('N6', 'spam', 0.21),
    ('N7', 'legitime', 0.42),
    ('N8', 'spam', 0.77),
    ('N9', 'legitime', 0.13),
    ('N10', 'spam', 0.58)
], columns=['message', 'label', 'score'])

# D5.E3 — vier Nachrichten, von zwei Filtern bewertet (Aufgabe 5.3)
four = pd.DataFrame([
    ('P1', 'spam', 0.80, 0.99),
    ('P2', 'spam', 0.60, 0.90),
    ('P3', 'legitime', 0.40, 0.10),
    ('P4', 'legitime', 0.20, 0.01)
], columns=['message', 'label', 'p_filter_A', 'p_filter_B'])

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `message` | Kennung der Nachricht (M1…) | — | — |
| `label` | tatsächliches Label: Spam oder legitim | — | `etiquette` |
| `links` | Anzahl der Links in der Nachricht | Links | `liens` |
| `suspicious_words` | Anzahl der verdächtigen Wörter in der Nachricht | Wörter | `mots_suspects` |
| `score` | geschätzte Spam-Wahrscheinlichkeit, auf zwei Dezimalstellen gerundet | — | — |
| `bin_min` | untere Grenze des Wahrscheinlichkeitsintervalls | — | `tranche_min` |
| `bin_max` | obere Grenze des Wahrscheinlichkeitsintervalls | — | `tranche_max` |
| `messages` | Anzahl der Nachrichten | Nachrichten | — |
| `spams` | Anzahl der Spams | Nachrichten | — |
| `mean_proba` | mittlere vorhergesagte Wahrscheinlichkeit im Intervall | — | `proba_moyenne` |
| `p_filter_A` | von Filter A angegebene Spam-Wahrscheinlichkeit | — | `p_filtre_A` |
| `p_filter_B` | von Filter B angegebene Spam-Wahrscheinlichkeit | — | `p_filtre_B` |
| `threshold` | Entscheidungsschwellenwert | — | `seuil` |
| `twelve` | die zwölf Nachrichten aus Kapitel 5 | — | — |

## Die Berechnungen der Lektionen, in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 5.1 bis 5.6, einschließlich der Likelihood (Ende von Lektion 5.4) und der Softmax (Lektion 5.6), mit ihren kurzen Fragen.

In [ ]:
# Exakte Werte der zwölf Nachrichten (Gleichungen 5.3 und 5.4): Auf sie wird die Regel angewendet
w, b = np.array([1, 0.5]), -4
X = twelve[['links', 'suspicious_words']].to_numpy()
z = X @ w + b
p_exact = sigmoid(z)
is_spam = (twelve['label'] == 'spam').to_numpy()
y = is_spam.astype(int)                 # 1 = Spam, 0 = legitim

# Lektion 5.1 — den Schwellenwert verschieben (Gleichungen 5.1 und 5.2)
for threshold in (0.5, 0.25):
    flagged = p_exact >= threshold                    # Regel: größer ODER GLEICH, auf den exakten Wert
    assert (flagged == (twelve['score'] >= threshold)).all()   # hier ändert die Rundung keine Entscheidung
    tp, fn = int((flagged & is_spam).sum()), int((~flagged & is_spam).sum())
    fp, tn = int((flagged & ~is_spam).sum()), int((~flagged & ~is_spam).sum())
    print(f'Schwellenwert {threshold}: VP = {tp}, FN = {fn}, FP = {fp}, VN = {tn} | '
          f'Trefferquote = {tp}/{tp + fn} = {tp / (tp + fn):.3f}, '            # Nenner: die Zeile der Spams
          f'Fehlalarme = {fp}/{fp + tn} = {fp / (fp + tn):.3f}, '   # Nenner: die Zeile der legitimen
          f'Präzision = {tp}/{tp + fp} = {tp / (tp + fp):.3f}')         # Nenner: die Spalte der markierten
switched = twelve['message'][(p_exact >= 0.25) & (p_exact < 0.5)]
print('Nachrichten, die von 0,5 zu 0,25 wechseln:', list(switched))

fig, axes = plt.subplots(2, 1, figsize=(7, 3.4), sharex=True)
for ax, threshold in zip(axes, (0.5, 0.25)):
    for q, s in zip(p_exact, is_spam):
        sig = q >= threshold
        ax.scatter(q, 1 if s else 0, s=70, marker='s' if s else 'o', color=SPAM if s else LEGIT,
                   edgecolor=FLAGGED if sig else 'gray', linewidth=2.4 if sig else 0.6)
    ax.axvline(threshold, color=THRESHOLD, linewidth=2.5)
    ax.set_yticks([0, 1], ['legitim', 'Spams']); ax.set_ylim(-0.6, 1.6)
    ax.set_title(f'Schwellenwert {threshold}', loc='left', color=THRESHOLD)
axes[1].set_xlabel('Geschätzte Wahrscheinlichkeit (exakter Wert)'); plt.tight_layout(); plt.show()

In [ ]:
# Lektion 5.2 — Rohwert, geschätzte Wahrscheinlichkeit, Entscheidung; der Fall M3; Kalibrierung
table = pd.DataFrame({'message': twelve['message'], 'z': z, 'p_exact': p_exact.round(4),
                        'p gerundet': p_exact.round(2),
                        'Entscheidung (Schwelle 0,5)': np.where(p_exact >= 0.5, 'markiert', 'nein')})
print(table.to_string(index=False))
print('gerundet = Werte aus Lektion 5.1:', np.allclose(p_exact.round(2), twelve['score']))
print(f"M3: z = {z[2]}, e^0 = {np.exp(0.0)}, p̂ = 1/(1 + 1) = {p_exact[2]} genau → "
      f"{'markiert' if p_exact[2] >= 0.5 else 'nicht markiert'} beim Schwellenwert 0,5")
print(f'M4: e^(-0,5) = {np.exp(-0.5):.4f}, Sigmoid(0,5) = {sigmoid(0.5):.3f}; M7: Sigmoid(-0,5) = {sigmoid(-0.5):.3f}')
print('Schwellenwert 0,5 für p̂ ⇔ Schwellenwert 0 für z:', bool(((p_exact >= 0.5) == (z >= 0)).all()),
      f'| Schwellenwert 0,25 für p̂ ⇔ z ≥ {np.log(0.25 / 0.75):.2f}')

zs = np.linspace(-5, 4.5, 300)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoid(zs), color=ORANGE, linewidth=2.2)
plt.axhline(0.5, color=THRESHOLD, linewidth=2)
plt.scatter(z[is_spam], p_exact[is_spam], marker='s', color=SPAM, zorder=3, label='Spams')
plt.scatter(z[~is_spam], p_exact[~is_spam], marker='o', color=LEGIT, edgecolor='gray', zorder=3, label='legitim')
plt.xlabel('Rohwert z'); plt.ylabel('Geschätzte Wahrscheinlichkeit p̂'); plt.legend(); plt.grid(alpha=0.4); plt.show()

# Eine geschätzte Wahrscheinlichkeit lässt sich prüfen: nebeneinander mittleres vorhergesagtes p̂, beobachtete Häufigkeit, Anzahl (Nenner)
cal = calibration.copy()
cal['beobachtete Häufigkeit'] = (cal['spams'] / cal['messages']).round(2)
print(cal[['bin_min', 'bin_max', 'mean_proba', 'beobachtete Häufigkeit', 'messages', 'spams']].to_string(index=False))
t = cal.iloc[4]
print(f"Intervall 0,7–0,9: vorhergesagt {t['mean_proba']:.2f}, beobachtet {int(t['spams'])} / {int(t['messages'])} ≈ "
      f"{t['spams'] / t['messages']:.2f} | erwartet {t['mean_proba'] * t['messages']:.0f} Spams, "
      f"Standardabweichung {np.sqrt(t['messages'] * 0.8 * 0.2):.1f}")
print('größte Abweichung zwischen vorhergesagt und beobachtet:', f"{np.max(np.abs(cal['mean_proba'] - cal['spams'] / cal['messages'])):.3f}")

In [ ]:
# Lektion 5.3 — die Strafe für einen echten Spam: −ln(p̂) (Gleichung 5.5). np.log ist der NATÜRLICHE Logarithmus.
for p in (0.90, 0.50, 0.10, 0.01):
    print(f'p̂ = {p:.2f} → Strafe = {-np.log(p):.3f}')
print(f'Verhältnisse: {np.log(0.01) / np.log(0.5):.1f} mal die Variante mit 0,50, '
      f'{np.log(0.01) / np.log(0.9):.1f} mal die Variante mit 0,90')
p_spams = p_exact[is_spam]                             # NICHT gerundete Werte aus Lektion 5.2 (M4: 0,622)
penalties = -np.log(p_spams)
print('Spams', list(twelve.loc[is_spam, 'message']), ': Strafen', penalties.round(2), f'| Mittelwert = {penalties.mean():.2f}')
print(f'Produkt der Wahrscheinlichkeiten = {p_spams.prod():.3f}; −ln(Produkt) = {-np.log(p_spams.prod()):.2f} '
      f'= Summe der Strafen = {penalties.sum():.2f}')
ps = np.linspace(0.006, 1, 300)
plt.figure(figsize=(6, 3))
plt.plot(ps, -np.log(ps), color=BLUE, linewidth=2.2)
plt.scatter([0.9, 0.5, 0.01], -np.log([0.9, 0.5, 0.01]), marker='s', color=SPAM, zorder=3)
plt.xlabel('Geschätzte Wahrscheinlichkeit p̂ für einen echten Spam'); plt.ylabel('Strafe −ln(p̂)'); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lektion 5.4 — der Verlust jeder Nachricht (Gleichung 5.6) und der mittlere Verlust (Gleichung 5.7)
def loss(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))   # −ln der Wahrscheinlichkeit, die der richtigen Antwort gegeben wird

p_tab = twelve['score'].to_numpy()                       # in der Tabelle angezeigte Werte (auf zwei Dezimalstellen)
losses = loss(y, p_exact)                              # Verluste aus den NICHT gerundeten Werten berechnet, wie im Buch
print(pd.DataFrame({'message': twelve['message'], 'y': y, 'p̂': p_tab,
                    'richtige Antwort': np.where(y == 1, p_tab, 1 - p_tab), 'loss': losses.round(2)}).to_string(index=False))
print(f'M4: −ln({p_exact[3]:.3f}) = {losses[3]:.3f} ≈ {losses[3]:.2f} (mit dem gerundeten 0,62 erhielte man {-np.log(0.62):.2f})')
print(f'M3: −ln(1 − 0,50) = {-np.log(0.5):.3f}; M6: −ln(1 − 0,27) = {-np.log(0.73):.3f}; M11: {-np.log(0.98):.3f}')
print(f'Summe der gerundeten Verluste = {losses.round(2).sum():.2f}; mittlerer Verlust = {losses.mean():.2f} '
      f'(ungerundete Summe: {losses.sum():.3f})')

# Zuerst anpassen: der mittlere Verlust als Funktion von b (w fest), wie die Verlustkurve aus Kapitel 3
def loss_b(bb):
    return loss(y, sigmoid(X @ w + bb)).mean()
for bb in (-5, -4, -3):
    print(f'b = {bb}: mittlerer Verlust = {loss_b(bb):.2f}')
bs = np.linspace(-7, -1, 6001)
Lb = [loss_b(bb) for bb in bs]
print(f'Tiefpunkt der Kurve: b ≈ {bs[int(np.argmin(Lb))]:.2f}, Verlust {min(Lb):.4f} (gegenüber {loss_b(-4):.4f} für b = −4)')
plt.figure(figsize=(6, 3))
plt.plot(bs, Lb, color=BLUE, linewidth=2.2)
plt.scatter([-4], [loss_b(-4)], marker='D', color=ORANGE, zorder=3)
plt.xlabel('Konstante b'); plt.ylabel('Mittlerer Verlust'); plt.grid(alpha=0.4); plt.show()
print('Quadratische Abweichung für den mit 0,01 bewerteten Spam:', round((1 - 0.01) ** 2, 2), '; für 0,50:', (1 - 0.5) ** 2)

In [ ]:
# Lektion 5.4 (Fortsetzung) — warum ein Logarithmus? Die Likelihood der zwölf Labels
# q = Wahrscheinlichkeit, die der Filter der TATSÄCHLICH beobachteten Klasse gibt: p̂ für einen Spam, 1 − p̂ für eine legitime Nachricht
q = np.where(y == 1, p_exact, 1 - p_exact)
V = q.prod()                                   # Nachrichten unabhängig bei gegebenem Modell: man multipliziert
print('q:', q.round(3))
print(f'Likelihood V = {V:.4f}')
print(f'−ln V = {-np.log(V):.3f} = Summe der zwölf Verluste {losses.sum():.3f}; '
      f'geteilt durch 12: {-np.log(V) / 12:.3f} = mittlerer Verlust')
print(f'Tausend Nachrichten dieser Qualität: V ≈ 10^{1000 / 12 * np.log10(V):.1f} (ohne Vorsichtsmaßnahme zu klein)')
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
axes[0].bar(twelve['message'], q, color=np.where(y == 1, SPAM, LEGIT))
axes[0].set_ylabel('q (beobachtete Klasse)'); axes[0].set_ylim(0, 1)
axes[1].barh(0, -np.log(q), left=np.concatenate([[0], np.cumsum(-np.log(q))[:-1]]),
             color=np.where(y == 1, SPAM, LEGIT), edgecolor='white')
axes[1].set_yticks([]); axes[1].set_xlabel('−ln q aneinandergereiht = −ln V')
plt.tight_layout(); plt.show()

In [ ]:
# Kurze Frage (Lektion 5.4) — Likelihood der acht legitimen Nachrichten
V_legit = q[y == 0].prod()   # 0,881 × 0,500 × 0,953 × 0,731 × 0,818 × 0,924 × 0,982 × 0,971

In [ ]:
# Prüfung der kurzen Frage
if V_legit is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Zeile.')
else:
    V_spams = q[y == 1].prod()
    print(f'Spams: {V_spams:.3f}; legitime: {V_legit:.3f}; Produkt: {V_spams * V_legit:.4f} (V = {V:.4f}) →',
          'richtig' if abs(V_legit - q[y == 0].prod()) < 1e-9 else 'bitte prüfen')
    print(f'in Logarithmen: {-np.log(V_spams):.3f} + {-np.log(V_legit):.3f} = {-np.log(V_spams) - np.log(V_legit):.3f}')

In [ ]:
# Lektion 5.5 — Schwellenwerte vergleichen: Alarme, Fehlalarme, verpasste, Präzision, Trefferquote, F1, Kosten (k = 3)
rows = []
for threshold in (0.25, 0.3, 0.5, 0.6, 0.7, 0.96):
    sig = p_exact >= threshold                              # exakte Werte
    tp, fp = int((sig & is_spam).sum()), int((sig & ~is_spam).sum())
    fn = int((~sig & is_spam).sum())
    precision = None if tp + fp == 0 else tp / (tp + fp)   # Spalte der markierten leer → undefiniert
    rows.append({'threshold': threshold, 'Alarme': tp + fp, 'Fehlalarme': fp, 'verpasst': fn,
                   'Präzision': show_value(precision), 'Trefferquote': f'{tp / (tp + fn):.2f}',
                   'F1': f'{2 * tp / (2 * tp + fp + fn):.2f}', 'Kosten (k = 3)': fn + 3 * fp})
print(pd.DataFrame(rows).to_string(index=False))
print(f'F1 beim Schwellenwert 0,3: 2 × 0,8 × 1 / 1,8 = {2 * 0.8 * 1 / 1.8:.2f}; beim Schwellenwert 0,6: 1,5 / 1,75 = {1.5 / 1.75:.2f}')

# Dieselbe Berechnung mit scikit-learn, falls installiert: Seine Konvention für eine undefinierte Präzision ist 0
try:
    from sklearn.metrics import precision_score, f1_score
    predicted_label = (p_exact >= 0.96).astype(int)
    print('scikit-learn beim Schwellenwert 0,96: Präzision =', precision_score(y, predicted_label, zero_division=0),
          '(Konvention zero_division=0); F1 =', f1_score(y, predicted_label, zero_division=0))
except ImportError:
    print('scikit-learn nicht installiert: Diese Berechnung ist optional.')

## Lektion 5.6 — Mehr als zwei Klassen: die Softmax

Drei Ordner, die einander ausschließen: Spam, legitime Werbung, gewöhnliche Nachricht. Die Softmax (Gleichung 5.10) verwandelt drei Werte in drei Wahrscheinlichkeiten mit der Summe 1; bei zwei Klassen ergibt sie wieder die Sigmoidfunktion (Gleichung 5.11).

In [ ]:
# Lektion 5.6 — die Softmax (Gleichung 5.10)
def softmax(scores):
    e = np.exp(np.asarray(scores, dtype=float))   # die Exponentialfunktion macht jeden Wert positiv und erhält die Reihenfolge
    return e / e.sum()                            # die Division durch die Summe liefert Wahrscheinlichkeiten mit der Summe 1

folders = ['Spam', 'legitime Werbung', 'gewöhnliche Nachricht']
scores_M3 = [2, 1, 0]                             # Newsletter M3; beobachtete Klasse: legitime Werbung
p3 = softmax(scores_M3)
print(pd.DataFrame({'Ordner': folders, 'z': scores_M3, 'e^z': np.exp(scores_M3).round(3), 'Wahrscheinlichkeit': p3.round(3)}).to_string(index=False))
print(f'Summe der Exponentialwerte = {np.exp(scores_M3).sum():.3f}; Summe der Wahrscheinlichkeiten = {p3.sum():.3f}')
print(f'Verlust von M3 = −ln {p3[1]:.3f} = {-np.log(p3[1]):.3f}')
p_better = softmax([0, 2, 0])
print(f'besser eingestellte Werte (0; 2; 0): Werbung {p_better[1]:.3f}, Verlust {-np.log(p_better[1]):.3f}')
# Zwei Klassen (Gleichung 5.11): legitim mit Wert 0, Spam mit Rohwert z
print(f'M7 (z = −0,5): Softmax mit zwei Klassen {softmax([-0.5, 0])[0]:.3f} = Sigmoid {sigmoid(-0.5):.3f}')
plt.figure(figsize=(6, 2.6))
plt.bar(folders, p3, color=[SPAM, BLUE, LEGIT], edgecolor=['none', 'black', 'none'], linewidth=2)
plt.ylabel('Wahrscheinlichkeit'); plt.ylim(0, 1); plt.title('Softmax der Werte (2; 1; 0) — beobachtete Klasse umrandet', loc='left')
plt.show()

In [ ]:
# Kurze Fragen (Lektion 5.6)
q1_probas = softmax([1, 1, 1])                                   # jede ist 1/3
q2_same = bool(np.allclose(softmax([3, 2, 1]), softmax([2, 1, 0])))   # überall +1: nichts ändert sich
q3_p_M4 = softmax([0.5, 0])[0]                                   # = Sigmoid(0,5 − 0) (Gleichung 5.11)

In [ ]:
# Prüfung der kurzen Fragen
if q1_probas is None or q2_same is None or q3_p_M4 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    print('1.', np.round(q1_probas, 3), f'; Verlust = ln 3 = {np.log(3):.3f} →',
          'richtig' if np.allclose(q1_probas, 1 / 3) else 'bitte prüfen')
    print('2.', q2_same, '→', 'richtig' if q2_same is True else 'bitte prüfen')
    print(f'3. {q3_p_M4:.3f} →', 'richtig' if abs(q3_p_M4 - sigmoid(0.5)) < 1e-9 else 'bitte prüfen')

## Aufgabe 5.1 — Ein Diagramm lesen: zwei Filter im Vergleich zur Diagonale

*Begriffe: geschätzte Wahrscheinlichkeit und Kalibrierung (Lektion 5.2).* Lesen Sie zuerst die Abbildung im Buch. Filter A: Daten D5.2; Filter B: Daten D5.E1.

### a) bis d) Kommentierte Lösung

Hier ist der Nenner jeder Häufigkeit **die Anzahl im Intervall**, nicht die Gesamtheit der 1000 Nachrichten.

**a)** Für Filter A beträgt die beobachtete Häufigkeit in dem Intervall, in dem er im Mittel 0,80 vorhersagt, etwa 0,79: Das ist die hervorgehobene Zeile der Tabelle aus Lektion 5.2 (71 Spams von 90 Nachrichten).

**b)** Filter B sagt in seinem letzten Intervall im Mittel 0,97 vorher, dort beobachtet man aber nur etwa 0,75 Spams. In seinem ersten Intervall sagt er 0,02 vorher, und man beobachtet 0,10.

**c)** Die Diagonale ist die perfekte Kalibrierung: beobachtete Häufigkeit = vorhergesagte Wahrscheinlichkeit. Die Punkte von A folgen ihr bis auf 0,02 genau. Die von B entfernen sich an beiden Enden davon: Oben kündigt B mehr Spams an, als es gibt (0,82 → 0,62; 0,97 → 0,75); unten kündigt er weniger an (0,02 → 0,10; 0,18 → 0,24). B ist **zu selbstsicher**: Seine Wahrscheinlichkeiten liegen zu nah an 0 und an 1.

**d)** Ja: Die beobachtete Häufigkeit von B steigt von Intervall zu Intervall (0,10; 0,24; 0,43; 0,55; 0,62; 0,75). Seine **Rangfolge** der Nachrichten bleibt also nützlich, und man kann weiterhin einen Schwellenwert mit einer Konfusionsmatrix wählen. Aber sein „0,97“ ist nicht als 97 Chancen von 100 zu lesen: Unter den so bewerteten Nachrichten beobachtet man etwa 3 Spams von 4.

In [ ]:
freq_b = (calibration_b['spams'] / calibration_b['messages']).to_numpy()   # Nenner: das Intervall
expected_last = calibration_b['mean_proba'].iloc[-1] * calibration_b['messages'].iloc[-1]   # 0,97 × 75

In [ ]:
# Überprüfung von e)
if freq_b is None or expected_last is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die vorherige Zelle.')
else:
    print('Häufigkeiten von B:', np.round(freq_b, 2), '→',
          'richtig' if np.allclose(freq_b, [0.10, 0.24, 0.43, 0.55, 0.62, 56 / 75]) else 'bitte prüfen')
    n, q = calibration_b['messages'].iloc[-1], calibration_b['mean_proba'].iloc[-1]
    sd = np.sqrt(n * q * (1 - q))                       # Standardabweichung der Anzahl Spams (Aufgabe 4.6)
    observed_freq = calibration_b['spams'].iloc[-1]
    print(f'Intervall 0,9–1: erwartet {expected_last:.2f} Spams, beobachtet {observed_freq}, Standardabweichung {sd:.2f}, '
          f'Abweichung = {(expected_last - observed_freq) / sd:.0f} Standardabweichungen →',
          'richtig' if abs(expected_last - 72.75) < 0.01 else 'bitte prüfen')
    plt.figure(figsize=(4.2, 4.2))
    plt.plot([0, 1], [0, 1], ':', color='gray', label='perfekte Kalibrierung')
    plt.plot(calibration['mean_proba'], calibration['spams'] / calibration['messages'], 'o-', color=BLUE, label='Filter A')
    plt.plot(calibration_b['mean_proba'], freq_b, '^--', color=GREEN, label='Filter B')
    plt.xlabel('Mittlere vorhergesagte Wahrscheinlichkeit'); plt.ylabel('Beobachtete Häufigkeit'); plt.legend(); plt.grid(alpha=0.4)
    plt.show()

### e) Kommentierte Lösung

Nehmen wir das Referenzmodell aus Lektion 5.2: unabhängige Labels, vorhergesagte Wahrscheinlichkeiten, die als korrekt angenommen werden (hier 0,97 für jede Nachricht im Intervall). Man würde $0{,}97 \times 75 \approx 72{,}75$ Spams erwarten. Man beobachtet 56. In diesem Modell beträgt die Standardabweichung der Anzahl Spams $\sqrt{75 \times 0{,}97 \times 0{,}03} \approx 1{,}48$ (Aufgabe 4.6). Die Abweichung, etwa 17 Spams, entspricht mehr als zehn Standardabweichungen: Das ist kein Zufall einer kleinen Stichprobe. 56 Spams sind mit diesem Modell nicht vereinbar, und Filter B ist wirklich zu selbstsicher. Für Filter A ergab derselbe Vergleich im Intervall 0,7–0,9 72 erwartete gegenüber 71 beobachteten Spams, bei einer Standardabweichung von 3,8: ein Ergebnis, das mit dem Referenzmodell vereinbar ist.

## Aufgabe 5.2 — Drei Schwellenwerte für einen Verein

*Begriffe: Entscheidungsregel und Konfusionsmatrix (Lektion 5.1), Trefferquote (Recall), Fehlalarmrate und Präzision (Gleichung 5.2), F1 und Kosten (Lektion 5.5).* Zehn vom Sekretariat geprüfte Nachrichten (Daten D5.E2), drei Schwellenwerte: 0,3, 0,5 und 0,7. **Die Wahrscheinlichkeiten in der Tabelle sind unverändert zu verwenden.** Bearbeiten Sie zuerst die Fragen a) bis f) auf Papier.

### a) bis c) Kommentierte Lösung

**a)** Auf der Achse, von links nach rechts: N2 (0,07, legitim), N9 (0,13, legitim), N6 (0,21, Spam), N5 (0,30, legitim), N7 (0,42, legitim), N4 (0,50, Spam), N10 (0,58, Spam), N3 (0,66, legitim), N8 (0,77, Spam), N1 (0,94, Spam). Zwei Nachrichten liegen genau auf einem Schwellenwert: N5 auf 0,3 und N4 auf 0,5. Da diese Dezimalzahlen die vom Filter gelieferten Werte sind, ist keine versteckte Rundung zu befürchten.

**b)** Die Regel „größer oder gleich“ markiert N4 beim Schwellenwert 0,5 und N5 beim Schwellenwert 0,3. Die Matrizen (Zeilen: was die Nachricht ist; Spalten: markiert, nicht markiert):

| Schwellenwert | VP (markierte Spams) | FN (verpasste Spams) | FP (Fehlalarme) | VN (durchgelassene legitime) |
|:--|--:|--:|--:|--:|
| 0,7 | 2 (N1, N8) | 3 (N10, N4, N6) | 0 | 5 |
| 0,5 | 4 (N1, N8, N10, N4) | 1 (N6) | 1 (N3) | 4 |
| 0,3 | 4 | 1 (N6) | 3 (N3, N7, N5) | 2 |

Kontrolle: In jeder Matrix gilt $VP + FN = 5$ Spams und $FP + VN = 5$ legitime Nachrichten.

**c)** Die Trefferquote (Recall) wird **in der Zeile der Spams** gezählt (5 Nachrichten), die Fehlalarmrate **in der Zeile der legitimen Nachrichten** (5 Nachrichten), die Präzision **in der Spalte der markierten**, deren Größe sich mit dem Schwellenwert ändert.

| Schwellenwert | Trefferquote | Fehlalarmrate | Präzision |
|:--|--:|--:|--:|
| 0,7 | 2/5 = 0,4 | 0/5 = 0 | 2/2 = 1 |
| 0,5 | 4/5 = 0,8 | 1/5 = 0,2 | 4/5 = 0,8 |
| 0,3 | 4/5 = 0,8 | 3/5 = 0,6 | 4/7 ≈ 0,57 |

Beim Schwellenwert 0,95 erreicht keine Nachricht den Schwellenwert (die höchste liegt bei 0,94): Die Spalte der markierten ist leer, ihre Anzahl ist 0, und die Präzision ist **undefiniert**. Sie ist weder 0 noch 1. Das Notebook zeigt sie als „undefiniert“ an; scikit-learn würde sie mit einer Warnung durch 0 ersetzen. Die Trefferquote dagegen ist 0.

### b) Die Konfusionsmatrix in Python

In [ ]:
def confusion_matrix(scores, is_spam, threshold):
    """scores und is_spam: zwei Listen gleicher Länge. Gibt das Wörterbuch
    {'VP': ..., 'FN': ..., 'FP': ..., 'VN': ...} zurück. Regel: markiert, wenn score >= threshold."""
    m = {'VP': 0, 'FN': 0, 'FP': 0, 'VN': 0}
    for s, spam in zip(scores, is_spam):
        flagged = s >= threshold                    # größer ODER GLEICH: N4 und N5 zählen
        if spam:
            m['VP' if flagged else 'FN'] += 1   # Zeile der Spams
        else:
            m['FP' if flagged else 'VN'] += 1   # Zeile der legitimen Nachrichten
    return m

In [ ]:
# Überprüfung: zuerst an den zwölf Nachrichten (EXAKTE Werte), dann an den zehn Nachrichten der Aufgabe
scenario = [('Lektion, Schwellenwert 0,5', list(p_exact), list(is_spam), 0.5, (3, 1, 1, 7)),
       ('Lektion, Schwellenwert 0,25', list(p_exact), list(is_spam), 0.25, (4, 0, 2, 6)),
       ('Aufgabe, Schwellenwert 0,7', list(ten['score']), list(ten['label'] == 'spam'), 0.7, (2, 3, 0, 5)),
       ('Aufgabe, Schwellenwert 0,5', list(ten['score']), list(ten['label'] == 'spam'), 0.5, (4, 1, 1, 4)),
       ('Aufgabe, Schwellenwert 0,3', list(ten['score']), list(ten['label'] == 'spam'), 0.3, (4, 1, 3, 2))]
for name, sc, sp, threshold, expected in scenario:
    m = confusion_matrix(sc, sp, threshold)
    if m is None:
        print(f'{name}: noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
    else:
        obtained = (m['VP'], m['FN'], m['FP'], m['VN'])
        print(f'{name}: VP, FN, FP, VN = {obtained} →', 'richtig' if obtained == expected else 'bitte prüfen')

### c) und e) Die Maße und der F1

In [ ]:
def metrics(m):
    def quotient(num, den):
        return None if den == 0 else num / den   # leere Gruppe: Das Maß ist undefiniert
    recall = quotient(m['VP'], m['VP'] + m['FN'])                 # Zeile der Spams
    false_alarm_rate = quotient(m['FP'], m['FP'] + m['VN'])   # Zeile der legitimen Nachrichten
    precision = quotient(m['VP'], m['VP'] + m['FP'])              # Spalte der markierten
    f1 = quotient(2 * m['VP'], 2 * m['VP'] + m['FP'] + m['FN'])   # auch ohne Alarm definiert
    return recall, false_alarm_rate, precision, f1

In [ ]:
# Überprüfung von c) und e)
scores_e2, spams_e2 = list(ten['score']), list(ten['label'] == 'spam')
if confusion_matrix(scores_e2, spams_e2, 0.5) is None:
    print('Ergänzen Sie zuerst confusion_matrix.')
elif metrics(confusion_matrix(scores_e2, spams_e2, 0.5))[0] is None:
    print('Maße: noch nicht ausgefüllt. Ergänzen Sie die Funktion metrics.')
else:
    expected_values = {0.7: (0.4, 0.0, 1.0), 0.5: (0.8, 0.2, 0.8), 0.3: (0.8, 0.6, 4 / 7)}
    for threshold, att in expected_values.items():
        r, f, p, _ = metrics(confusion_matrix(scores_e2, spams_e2, threshold))
        print(f'Schwellenwert {threshold}: Trefferquote = {show_value(r)}, Fehlalarme = {show_value(f)}, Präzision = {show_value(p)} →',
              'richtig' if np.allclose((r, f, p), att) else 'bitte prüfen')
    r, f, p, f1 = metrics(confusion_matrix(scores_e2, spams_e2, 0.95))
    print(f'Schwellenwert 0.95: kein Alarm; Trefferquote = {show_value(r)}, Präzision = {show_value(p)}, F1 = {show_value(f1)} →',
          'richtig' if r == 0 and p is None and f1 == 0 else 'bitte prüfen')
    f1s = [metrics(confusion_matrix(scores_e2, spams_e2, s))[3] for s in (0.7, 0.5, 0.3)]
    print('F1 bei den Schwellenwerten 0,7 / 0,5 / 0,3:', [round(v, 2) for v in f1s], '→',
          'richtig' if np.allclose(f1s, [4 / 7, 0.8, 8 / 12]) else 'bitte prüfen')

### f) Die Kosten der Fehler

In [ ]:
def cost(m, k):
    return m['FN'] + k * m['FP']   # ein Fehlalarm „wiegt“ so viel wie k verpasste Spams

In [ ]:
# Überprüfung von f)
if confusion_matrix(scores_e2, spams_e2, 0.5) is None or cost({'FN': 0, 'FP': 0}, 3) is None:
    print('Kosten: noch nicht ausgefüllt. Ergänzen Sie die Funktionen.')
else:
    for k, expected in ((3, [3, 4, 10]), (1, [3, 2, 4])):
        costs = [cost(confusion_matrix(scores_e2, spams_e2, s), k) for s in (0.7, 0.5, 0.3)]
        best_one = (0.7, 0.5, 0.3)[costs.index(min(costs))]
        print(f'k = {k}: Kosten bei den Schwellenwerten 0,7 / 0,5 / 0,3 = {costs}, bester Schwellenwert {best_one} →',
              'richtig' if costs == expected else 'bitte prüfen')

### g) Alle Werte des Schwellenwerts

In [ ]:
# Trefferquote, Präzision und F1 für die Schwellenwerte 0, 0,01, …, 1 (undefinierte Präzision: kein Punkt)
if confusion_matrix(scores_e2, spams_e2, 0.5) is None or metrics(confusion_matrix(scores_e2, spams_e2, 0.5))[0] is None:
    print('Ergänzen Sie confusion_matrix und metrics, um die Kurven zu zeichnen.')
else:
    thresholds = np.round(np.arange(0, 1.001, 0.01), 2)
    R, F, P, F1 = zip(*[metrics(confusion_matrix(scores_e2, spams_e2, s)) for s in thresholds])
    P = [np.nan if p is None else p for p in P]
    plt.figure(figsize=(6, 3.2))
    plt.step(thresholds, R, where='post', color=SPAM, linewidth=2, label='Trefferquote (Zeile der Spams)')
    plt.step(thresholds, P, where='post', color=FLAGGED, linewidth=2, linestyle='--', label='Präzision (Spalte der markierten)')
    plt.step(thresholds, F1, where='post', color='black', linewidth=1.5, linestyle=':', label='F1')
    for s in (0.3, 0.5, 0.7):
        plt.axvline(s, color=THRESHOLD, linewidth=1, alpha=0.6)
    plt.xlabel('Schwellenwert'); plt.ylim(-0.05, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### d) bis g) Kommentierte Lösung

**d)** Von 0,5 zu 0,3 wechseln zwei Nachrichten: N7 (0,42) und N5 (0,30, markiert, weil die Regel „größer **oder gleich**“ lautet). Beide sind **legitime** Nachrichten. Man gewinnt nichts: Kein zusätzlicher Spam wird erfasst, die Trefferquote bleibt bei 0,8, denn der einzige verpasste Spam, N6 (0,21), liegt weiterhin unter dem Schwellenwert. Man verliert zwei weitere Fehlalarme: Die Rate steigt von 0,2 auf 0,6, und die Präzision sinkt von 0,8 auf 4/7 ≈ 0,57.

**e)** Mit $F_1 = 2\,VP / (2\,VP + FP + FN)$: Schwellenwert 0,7: $4 / (4 + 0 + 3) = 4/7 \approx 0{,}57$; Schwellenwert 0,5: $8 / (8 + 1 + 1) = 0{,}8$; Schwellenwert 0,3: $8 / (8 + 3 + 1) = 8/12 \approx 0{,}67$. Der F1 zeigt auf den Schwellenwert **0,5**. Beim Schwellenwert 0,95 ohne Alarm ist die Präzision undefiniert, der F1 aber beträgt $0 / (0 + 0 + 5) = 0$.

**f)** Wenn ein Fehlalarm so viel kostet wie drei verpasste Spams: Schwellenwert 0,7: $3 + 3 \times 0 = 3$; Schwellenwert 0,5: $1 + 3 \times 1 = 4$; Schwellenwert 0,3: $1 + 3 \times 3 = 10$. Der Schwellenwert **0,7** ist am günstigsten: Er lässt drei Spams durch, verliert aber keine Nachricht des Vereins. Es ist nicht der Schwellenwert des F1. In $F_1 = 2\,VP / (2\,VP + FP + FN)$ treten $FP$ und $FN$ symmetrisch auf, mit demselben Koeffizienten. Von 0,7 zu 0,5 erfasst man zwei Spams mehr (N10 und N4) um den Preis eines Fehlalarms (N3): $FP + FN$ sinkt von 3 auf 2 und $VP$ steigt von 2 auf 4, also steigt der F1. Die Regel des Sekretariats gibt einem Fehlalarm drei Kosteneinheiten und jedem verpassten Spam eine: Die Kosten ändern sich um $-2 + 3 = +1$. Kosten beide Fehler gleich viel ($k = 1$), werden die Kosten zu 3, 2 und 4, und 0,5 gewinnt, wie beim F1. Die Wahl des Schwellenwerts hängt davon ab, was jeder Fehler kostet, und das muss das Sekretariat festlegen. Prüfen Sie zum Schluss das Ganze: Diese Maße beruhen auf **zehn Nachrichten**, einer viel zu kleinen Stichprobe, um einen endgültigen Schwellenwert festzulegen.

**g)** Wenn der Schwellenwert steigt, kann die Trefferquote nur sinken, stufenweise, jedes Mal, wenn ein Spam die Spalte der markierten verlässt. Die Präzision hat keine feste Änderungsrichtung, weil sich ihr Nenner bei jeder Stufe ändert. Oberhalb von 0,94 bricht die Präzisionskurve ab: Sie ist undefiniert, da es keinen Alarm mehr gibt. Der F1 dagegen fällt auf 0.

## Aufgabe 5.3 — Interpretieren: gleiche Entscheidungen, unterschiedliche Verluste

*Begriffe: logarithmischer Verlust (Lektionen 5.3 und 5.4), Entscheidungsregel (Gleichung 5.1).* Daten D5.E3: Diese Wahrscheinlichkeiten sind die zu verwendenden Werte.

### a) bis d) Kommentierte Lösung

**a)** Beim Schwellenwert 0,5 markieren beide Filter P1 und P2 und lassen P3 und P4 durch: kein Fehler, weder bei A noch bei B. Ihre Konfusionsmatrizen sind identisch.

**b)** Man nimmt $-\ln$ der Wahrscheinlichkeit, die der richtigen Antwort gegeben wird: $\hat p$ für einen Spam, $1 - \hat p$ für eine legitime Nachricht.

- Filter A: $-\ln(0{,}8) \approx 0{,}223$; $-\ln(0{,}6) \approx 0{,}511$; $-\ln(1 - 0{,}4) = -\ln(0{,}6) \approx 0{,}511$; $-\ln(1 - 0{,}2) = -\ln(0{,}8) \approx 0{,}223$. Mittelwert: $1{,}468 / 4 \approx 0{,}37$.
- Filter B: $-\ln(0{,}99) \approx 0{,}010$; $-\ln(0{,}9) \approx 0{,}105$; $-\ln(0{,}9) \approx 0{,}105$; $-\ln(0{,}99) \approx 0{,}010$. Mittelwert: $0{,}231 / 4 \approx 0{,}06$.

**c)** B ist vorzuziehen: Er trifft dieselben Entscheidungen, aber mit mehr Sicherheit, und das zu Recht. Die Konfusionsmatrix schaut nur, auf welche Seite des Schwellenwerts jede Wahrscheinlichkeit fällt; der Verlust schaut, wie weit sie von der richtigen Antwort entfernt ist. Deshalb passt man die Gewichte mit dem Verlust an und nicht mit der Anzahl der Fehler.

**d)** C unterscheidet sich von B nur bei P2, das mit 0,01 bewertet wird: $-\ln(0{,}01) \approx 4{,}605$. Sein mittlerer Verlust beträgt $(0{,}010 + 4{,}605 + 0{,}105 + 0{,}010) / 4 \approx 1{,}18$, mehr als dreimal so viel wie bei A. Dabei macht C beim Schwellenwert 0,5 nur einen Fehler (er verpasst P2). Ein einziger, selbstsicher begangener Fehler genügt, um den Verlust explodieren zu lassen: Das ist gewollt (Lektion 5.3).

In [ ]:
def mean_loss(labels, probas):
    """labels: Liste aus 1 (Spam) und 0 (legitim); probas: geschätzte Wahrscheinlichkeiten, Spam zu sein."""
    y_, p_ = np.array(labels), np.array(probas, dtype=float)
    return float(np.mean(-(y_ * np.log(p_) + (1 - y_) * np.log(1 - p_))))   # Mittelwert über die Nachrichten

In [ ]:
# Überprüfung
y4 = list((four['label'] == 'spam').astype(int))
if mean_loss(y4, four['p_filter_A']) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    p_c = [0.99, 0.01, 0.10, 0.01]
    for name, probas, att in (('A', four['p_filter_A'], 0.3670), ('B', four['p_filter_B'], 0.0577), ('C', p_c, 1.1827)):
        v = mean_loss(y4, probas)
        print(f'Filter {name}: mittlerer Verlust = {v:.4f} →', 'richtig' if abs(v - att) < 0.0001 else 'bitte prüfen')
    print('Kontrolle mit den zwölf Nachrichten aus Lektion 5.4:', round(mean_loss(list(y), list(p_exact)), 2))

## Aufgabe 5.4 — Die Kosten bestimmen den Schwellenwert

*Begriffe: Kosten eines Schwellenwerts (Gleichung 5.9), F1 (Gleichung 5.8).* Wir nehmen wieder die zwölf Nachrichten mit ihren **exakten** Wahrscheinlichkeiten `p_exact`: Auf sie wird die Regel angewendet.

### a) Erwartete Vorhersage

Der beste Schwellenwert **steigt**: Wenn Fehlalarme teuer sind, nimmt man in Kauf, Spams durchzulassen, um weniger Alarme auszulösen.

In [ ]:
def total_cost(threshold, k):
    flagged = p_exact >= threshold                    # EXAKTE Werte: M6 ist 0,2689…, nicht 0,27
    missed = int((~flagged & is_spam).sum())    # Zeile der Spams
    false_alarms = int((flagged & ~is_spam).sum())    # Zeile der legitimen Nachrichten
    return missed + k * false_alarms

In [ ]:
# b) bis e) Die besten Schwellenwerte für jedes k, dann die Schwellenwerte des F1
thresholds = np.round(np.arange(0, 1.001, 0.01), 2)
if total_cost(0.5, 3) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    expected_values = {0.1: (0.27, 0.37), 1: (0.27, 0.62), 3: (0.51, 0.62), 10: (0.51, 0.62)}
    plt.figure(figsize=(6, 3.2))
    for k, att in expected_values.items():
        c = np.array([total_cost(s, k) for s in thresholds])
        best_ones = thresholds[np.isclose(c, c.min())]
        print(f'k = {k:>4}: minimale Kosten {c.min():.1f}, erreicht für die Schwellenwerte von {best_ones.min():.2f} bis {best_ones.max():.2f} →',
              'richtig' if (best_ones.min(), best_ones.max()) == att else 'bitte prüfen')
        plt.step(thresholds, c / c.max(), where='post', label=f'k = {k}')
    plt.xlabel('Schwellenwert'); plt.ylabel('Kosten (geteilt durch ihr Maximum)'); plt.legend(fontsize=8); plt.grid(alpha=0.4)
    plt.show()
f1 = []
for s in thresholds:
    sig = p_exact >= s
    tp, fp, fn = (sig & is_spam).sum(), (sig & ~is_spam).sum(), (~sig & is_spam).sum()
    f1.append(2 * tp / (2 * tp + fp + fn))
f1 = np.array(f1)
best = thresholds[np.isclose(f1, f1.max())]
print(f'maximaler F1 {f1.max():.2f}, für die Schwellenwerte von {best.min():.2f} bis {best.max():.2f}')
print('M6: exakter Wert', round(float(p_exact[5]), 4), '→ markiert beim Schwellenwert 0,27?', bool(p_exact[5] >= 0.27))

### c) bis e) Kommentierte Lösung

**c)** Der beste Schwellenwert steigt tatsächlich mit $k$: von 0,27–0,37 für $k = 0{,}1$ auf 0,51–0,62 für $k = 3$ oder 10. Oberhalb von $k = 3$ bewegt er sich nicht mehr: Der einzige verbleibende Fehlalarm, M3 (0,5), ist bereits vermieden, und ein noch höherer Schwellenwert würde nur Spams verpassen.

**d)** Zwischen zwei benachbarten geschätzten Wahrscheinlichkeiten wechselt keine Nachricht die Kategorie: Die Konfusionsmatrix und damit die Kosten sind für alle Schwellenwerte des Intervalls gleich. Für $k = 1$ kosten die Schwellenwerte von 0,27 bis 0,37 (ein Fehlalarm, kein verpasster Spam) und von 0,51 bis 0,62 (kein Fehlalarm, ein verpasster Spam) alle 1; dazwischen, von 0,38 bis 0,50, liegen M3 und M7 auf der falschen Seite, und die Kosten betragen 2. M6 wird beim Schwellenwert 0,27 nicht markiert, weil sein exakter Wert 0,2689… ist: Sein „0,27“ ist nur ein gerundeter Wert.

**e)** Der F1 ist maximal (0,89) für die Schwellenwerte von 0,27 bis 0,37. Die Kosten zeigen auf dieselben Schwellenwerte, wenn Fehlalarme billig sind ($k = 0{,}1$), und schließen sie für $k = 1$ gleichauf ein. Sobald $k$ den Wert 3 erreicht, gehen Kosten und F1 auseinander: Das ist die Lage des Vereins (Lektion 5.5).

## Aufgabe 5.5 — Vorhersagen, dann prüfen: eine Rundung am Rand des Schwellenwerts

*Begriffe: Entscheidungsregel (Gleichung 5.1), Sigmoidfunktion (Gleichung 5.4), F1 (Gleichung 5.8).* Gleiche Gewichte, aber eine Konstante b = −4,01.

### a) bis c) und e) Kommentierte Lösung

**a)** Eine häufige Vorhersage: „Nichts ändert sich, eine Senkung um 0,01 ist winzig.“ Auf zwei Dezimalstellen gerundet sind die zwölf Wahrscheinlichkeiten tatsächlich dieselben wie in der Tabelle aus Lektion 5.2. Trotzdem wechselt M3 die Seite.

**b)** $z = 1 \times 4 + 0{,}5 \times 0 - 4{,}01 = -0{,}01$. Dann $e^{0{,}01} \approx 1{,}01005$ und $\hat p = 1 / 2{,}01005 \approx 0{,}4975$. Auf zwei Dezimalstellen gerundet: 0,50. Aber $0{,}4975 < 0{,}5$: M3 **wird nicht markiert**. Wer in einer gerundeten Tabelle 0,50 liest, käme zur falschen Entscheidung.

**c)** Beim Schwellenwert 0,5: $VP = 3$, $FN = 1$ (M7), $FP = 0$, $VN = 8$. Präzision $3/3 = 1$, Trefferquote $3/4 = 0{,}75$, $F_1 = 6/7 \approx 0{,}86$, wie beim Schwellenwert 0,6 in der Tabelle aus Lektion 5.5: Der Fehlalarm M3 ist verschwunden. Beim Schwellenwert 0,25 ändert sich nichts: M6 liegt jetzt bei 0,2670…, immer noch über 0,25.

**e)** Die Regel vergleicht den vom Filter berechneten Wert mit dem Schwellenwert, nicht seine Darstellung. Eine Rundung kann den Schwellenwert in die eine oder die andere Richtung überschreiten lassen. Wer gerundete Wahrscheinlichkeiten veröffentlicht, muss daher angeben, ob die Entscheidungen auf den exakten Werten getroffen wurden, und die Fälle am Rand des Schwellenwerts kennzeichnen.

In [ ]:
p_m3 = float(sigmoid(1 * 4 + 0.5 * 0 - 4.01))   # z = −0,01 → 0,4975…
flagged_m3 = p_m3 >= 0.5                          # False: Der exakte Wert liegt unter dem Schwellenwert

In [ ]:
# Überprüfung, dann d) Entscheidungen auf den exakten und auf den gerundeten Werten
if p_m3 is None or flagged_m3 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die vorherige Zelle.')
else:
    print(f'M3: p̂ = {p_m3:.4f} (angezeigt {p_m3:.2f}), markiert: {flagged_m3} →',
          'richtig' if abs(p_m3 - 0.4975) < 0.0001 and flagged_m3 is False else 'bitte prüfen')
p_coll = sigmoid(X @ w - 4.01)
for threshold in (0.5, 0.25):
    exact, rounded = p_coll >= threshold, p_coll.round(2) >= threshold
    diff = list(twelve['message'][exact != rounded])
    print(f'Schwellenwert {threshold}: unterschiedliche Entscheidungen zwischen exaktem und gerundetem Wert: {diff if diff else "keine"}')
sig = p_coll >= 0.5
tp, fp, fn = int((sig & is_spam).sum()), int((sig & ~is_spam).sum()), int((~sig & is_spam).sum())
print(f'Kollege, Schwellenwert 0,5: VP = {tp}, FN = {fn}, FP = {fp}; Präzision = {tp / (tp + fp):.2f}, '
      f'Trefferquote = {tp / (tp + fn):.2f}, F1 = {2 * tp / (2 * tp + fp + fn):.2f}')

## Aufgabe 5.6 — Herausforderung: den logarithmischen Verlust hinabsteigen

*Begriffe: mittlerer Verlust (Gleichung 5.7), partielle Ableitungen und Gradient (Gleichung 3.6, Lektion 3.4), Sigmoidfunktion (Gleichung 5.4).*

### a) bis c) Kommentierte Lösung

**a)** Für einen Spam ($y = 1$) beträgt der Verlust $-\ln(\sigma(z))$. Seine Ableitung ist $-\sigma'(z) / \sigma(z) = -(1 - \sigma(z)) = \hat p - 1 = \hat p - y$. Für eine legitime Nachricht ($y = 0$) beträgt der Verlust $-\ln(1 - \sigma(z))$, mit der Ableitung $\sigma'(z) / (1 - \sigma(z)) = \sigma(z) = \hat p - 0 = \hat p - y$. In beiden Fällen: $\hat p - y$.

**b)** Da $z_i = w_1 x_{i1} + w_2 x_{i2} + b$, ist die Ableitung von $z_i$ nach $b$ gleich 1 und nach $w_j$ gleich $x_{ij}$. Gemittelt über die Nachrichten: $\partial L / \partial b = \frac{1}{n} \sum (\hat p_i - y_i)$ und $\partial L / \partial w_j = \frac{1}{n} \sum (\hat p_i - y_i)\, x_{ij}$. Das ist dieselbe Form wie beim MSE (Gleichung 3.6): eine **Abweichung zwischen Vorhersage und Wirklichkeit**, multipliziert mit dem Merkmal. $\hat p_i - y_i$ spielt die Rolle der Abweichung aus Kapitel 3, bis auf das Vorzeichen und den Faktor 2.

**c)** Mit den gerundeten Wahrscheinlichkeiten aus Lektion 5.2: $\sum \hat p_i = 4{,}08$, und es gibt 4 Spams, also $\partial L / \partial b = (4{,}08 - 4) / 12 \approx 0{,}007$ (0,006 mit den exakten Werten). Das positive Vorzeichen sagt, dass man $b$ ein wenig senken sollte; die fast verschwindende Größe sagt, dass $b = -4$ schon fast am Tiefpunkt der Verlustkurve aus Lektion 5.4 liegt, deren Minimum bei etwa $-4{,}05$ liegt. Am Tiefpunkt ist die Summe der geschätzten Wahrscheinlichkeiten gleich der Anzahl der Spams.

In [ ]:
def gradient(w_, b_):
    """Gibt (dL/dw, dL/db) zurück: dL/dw ist ein Array aus zwei Zahlen."""
    gap = sigmoid(X @ w_ + b_) - y          # p̂ − y: die „Abweichung“ jeder Nachricht
    return X.T @ gap / len(y), gap.mean()  # Mittelwert von gap × Merkmal, Mittelwert von gap

In [ ]:
# Überprüfung, dann d) 1000 Gradientenschritte (η = 0,5) ab w = (1; 0,5), b = −4
def avg_loss(w_, b_):
    p_ = sigmoid(X @ w_ + b_)
    return float(np.mean(-(y * np.log(p_) + (1 - y) * np.log(1 - p_))))

if gradient(w, b) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    gw, gb = gradient(w, b)
    print(f'zu Beginn: dL/db = {gb:.4f}, dL/dw = {np.round(gw, 4)} →',
          'richtig' if abs(gb - 0.0062) < 0.0001 and np.allclose(gw, [0.0144, -0.2368], atol=0.0001) else 'bitte prüfen')
    w_, b_ = w.astype(float), float(b)
    for step_k in range(1, 1001):
        gw, gb = gradient(w_, b_)
        w_, b_ = w_ - 0.5 * gw, b_ - 0.5 * gb
        if step_k in (1, 10, 100, 1000):
            print(f'nach {step_k:>4} Schritten: Verlust = {avg_loss(w_, b_):.4f}, w = {np.round(w_, 2)}, b = {b_:.2f}')
    print('Endverlust →', 'richtig' if abs(avg_loss(w_, b_) - 0.0308) < 0.0005 else 'bitte prüfen')
# e) Gewichte, die die zwölf Nachrichten perfekt trennen: verdächtige Wörter ≥ 3 bei Spams, ≤ 2 bei legitimen
for factor in (1, 10):
    print(f'w = (0 ; {2 * factor}), b = {-5 * factor}: mittlerer Verlust = {avg_loss(np.array([0, 2.0]) * factor, -5.0 * factor):.6f}')

### d) und e) Kommentierte Lösung

**d)** Der mittlere Verlust sinkt nach 1000 Schritten von 0,26 auf etwa 0,031 und würde weiter sinken. Die Gewichte dagegen ändern sich stark: Das Gewicht der verdächtigen Wörter steigt auf etwa 4,4, die Konstante sinkt auf etwa −9,2, und das Gewicht der Links wird **negativ** (etwa −0,54). Das Modell hat bemerkt, dass M3, der Newsletter, viele Links hat und kein Spam ist. Bei zwölf Nachrichten ist das ein Zufall, keine Regel: Nichts spricht dafür, dass ein Link eine Nachricht weniger verdächtig macht.

**e)** Mit $w = (0 ; 2)$ und $b = -5$ haben alle Spams $z \geq 1$ und alle legitimen Nachrichten $z \leq -1$: Der mittlere Verlust beträgt etwa 0,10. Mit zehnmal größeren Gewichten fällt er auf etwa 0,000 01. Er kann sich 0 beliebig nähern, ohne es je zu erreichen: Die zwölf Nachrichten sind **trennbar**. Aber ein so selbstsicherer Filter wäre bei einem Spam der nächsten Woche, der nur zwei verdächtige Wörter hätte, katastrophal: Sein Verlust wäre riesig. Deshalb fügt scikit-learn eine Regularisierung hinzu, die das Wachstum der Gewichte bremst. Und das ist die Frage von Kapitel 6: Ein Verlust, der auf den zur Anpassung verwendeten Nachrichten gemessen wurde, sagt nicht, was bei den anderen geschehen wird.